In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.stattools import adfuller, kpss
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import arch.unitroot as au
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import friedmanchisquare
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.stats.diagnostic import het_white, het_breuschpagan
from statsmodels.tsa.ardl import ARDL, ardl_select_order
import sys
import os
os.environ['RPY2_CFFI_MODE'] = 'ABI'
from scipy.stats import shapiro
from statsmodels.stats.stattools import durbin_watson
import itertools
import math
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS
from scipy.stats import chi2, f
from statsmodels.tsa.x13 import x13_arima_analysis
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri, globalenv
from rpy2.robjects.packages import importr, isinstalled
from rpy2.robjects.conversion import localconverter
import rpy2.rinterface_lib.callbacks
import os
import pickle

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
###############################
    #ЗАГРУЗКА + ПРЕДОБРАБОТКА ДАННЫХ
###############################
df_reg = pd.read_excel('База данных_рег и фед показатели.xlsx', skiprows=1, sheet_name = 'data')
df_RF = pd.read_excel('База данных_рег и фед показатели.xlsx', skiprows=1, sheet_name = 'data_RF')
households_structure = pd.read_excel('База_соц эконом неоднор регионов.xlsx', skiprows=1, sheet_name = 'Фин пол ДХ')
households_loans = pd.read_excel('База_соц эконом неоднор регионов.xlsx', sheet_name = 'КЖН (рейтинг)')
funds_coefficient = pd.read_excel('База_соц эконом неоднор регионов.xlsx', skiprows=2, sheet_name = 'Коэф фондов')

In [ ]:
# Преобразование даты 
df_RF['Date'] = pd.to_datetime(df_RF['Date'])
df_RF = df_RF.sort_values('Date')
df_reg['Date'] = pd.to_datetime(df_reg['Date'])
df_reg = df_reg.sort_values(['Num_reg','Date'])

In [ ]:
df_reg.head()

In [ ]:
df_RF.head()

In [ ]:
###############################
# Аанализ панельных данных
###############################
pickle_filename = 'mon_shock_dataset.pkl'
with open(pickle_filename, 'rb') as f:
    df_shocks = pickle.load(f)
df_shocks['Date'] = pd.to_datetime(df_shocks['Date'])

if isinstance(df_reg.index, pd.MultiIndex):
    df_reg = df_reg.reset_index() 

# Преобразуем Date
df_reg['Date'] = pd.to_datetime(df_reg['Date'])

# ============ ФЕДЕРАЛЬНЫЕ ПЕРЕМЕННЫЕ ============

df_reg = df_reg.merge(df_shocks[['Date', 'Mon_Shock']], 
                      on='Date', 
                      how='left')

df_exc = df_RF[['Date', 'exc_rate']].copy()
df_reg = df_reg.merge(df_exc, on='Date', how='left')

df_infl = df_RF[['Date', 'Inflation_Expectations']].copy()
df_reg = df_reg.merge(df_infl, on='Date', how='left')

df_bond = df_RF[['Date', 'Bonds_Rate_Correct_5Y']].copy()
df_reg = df_reg.merge(df_bond, on='Date', how='left')

# Переименование
df_reg.rename(columns={'exc_rate': 'Exc_rate'}, inplace=True)

# Создаем лаги
df_reg['Int_Rate_FL_lag1'] = df_reg.groupby('Region')['Int_Rate_FL'].shift(1)
df_reg['Int_Rate_Mort_lag1'] = df_reg.groupby('Region')['Int_Rate_Mort'].shift(1)
df_reg['Int_Rate_ConsCred_lag1'] = df_reg.groupby('Region')['Int_Rate_ConsCred'].shift(1)

# СПИСОК ПЕРЕМЕННЫХ
initial_vars = [
    'Int_Rate_FL',
    'Int_Rate_FL_lag1',
    'Int_Rate_Mort',
    'Credit_impulse',
    'Int_Rate_Mort_lag1',
    'Int_Rate_ConsCred',
    'Int_Rate_ConsCred_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Cred_structure',
    'Def_Zadolg_Fl',
    'Def_Zadolg_Mort',
    'Def_Zadolg_ConsCred',
   # 'New_Loans_Progr',
    'New_Loans_Fl',
    'New_Loans_Mort',
    'New_Loans_ConsCred',
    'Mon_Shock',
    'Exc_rate',
    'Inflation_Expectations',
    'Bonds_Rate_Correct_5Y'
]

df_reg_analys = df_reg[['Region', 'Date'] + initial_vars].copy()
df_reg_analys = df_reg_analys[df_reg['Region'] != 'Москва']
df_reg_analys = df_reg_analys.dropna()

In [ ]:
df_reg_analys.head()

In [ ]:
df_reg_analys.info()

In [ ]:
print("\n" + "="*70)
print("ОЧИСТКА РЕГИОНОВ ПО МЕТОДУ IQR")
print("="*70)

key_vars = [
    'Int_Rate_FL',
    'Int_Rate_FL_lag1',
    'Int_Rate_Mort',
    'Credit_impulse',
    'Int_Rate_Mort_lag1',
    'Int_Rate_ConsCred',
    'Int_Rate_ConsCred_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Cred_structure',
    'Def_Zadolg_Fl',
    'Def_Zadolg_Mort',
    'Def_Zadolg_ConsCred',
   # 'New_Loans_Progr',
    'New_Loans_Fl',
    'New_Loans_Mort',
    'New_Loans_ConsCred',
    'Mon_Shock',
    'Exc_rate',
    'Inflation_Expectations',
    'Bonds_Rate_Correct_5Y'
]

def detect_outlier_regions_iqr(df, key_vars, min_obs=60, max_na_pct=0.3):
    """
    Определение выбросов по IQR (межквартильный размах)
    """
    region_stats = []
    
    for region in df['Region'].unique():
        reg_data = df[df['Region'] == region][key_vars].dropna()
        
        # 1. Минимум наблюдений
        n_obs = len(reg_data)
        if n_obs < min_obs:
            region_stats.append({
                'Region': region,
                'N_obs': n_obs,
                'Reason': f'Мало данных (<{min_obs})',
                'Status': 'EXCLUDE'
            })
            continue
        
        # 2. % пропусков
        total_rows = len(df[df['Region'] == region])
        na_pct = 1 - (n_obs / total_rows)
        if na_pct > max_na_pct:
            region_stats.append({
                'Region': region,
                'N_obs': n_obs,
                'NA_pct': f'{na_pct:.1%}',
                'Reason': f'Много пропусков (>{max_na_pct*100}%)',
                'Status': 'EXCLUDE'
            })
            continue
        
        # 3. IQR по переменным
        iqr_outliers = []
        for col in key_vars:
            if col in reg_data.columns:
                data = reg_data[col].dropna()
                if len(data) > 0:
                    Q1 = data.quantile(0.15)
                    Q3 = data.quantile(0.85)
                    IQR = Q3 - Q1
                    lower_bound = Q1 - 1.5 * IQR
                    upper_bound = Q3 + 1.5 * IQR
                    outliers = ((data < lower_bound) | (data > upper_bound)).sum()
                    iqr_outliers.append(outliers)
        
        iqr_total = sum(iqr_outliers)
        iqr_pct = iqr_total / n_obs if n_obs > 0 else 0

        
        # РЕШЕНИЕ
        status = 'KEEP'
        reasons = []
        
        if iqr_pct > 0.2:  # >20% выбросов по IQR
            reasons.append(f'Много выбросов по IQR ({iqr_pct:.1%})')
            status = 'EXCLUDE'
        
        region_stats.append({
            'Region': region,
            'N_obs': n_obs,
            'NA_pct': f'{na_pct:.1%}',
            'IQR_outliers': iqr_total,
            'IQR_pct': f'{iqr_pct:.1%}',
            'Reasons': '; '.join(reasons),
            'Status': status
        })
    
    return pd.DataFrame(region_stats)

# АНАЛИЗ РЕГИОНОВ ПО IQR
region_quality_iqr = detect_outlier_regions_iqr(df_reg_analys, key_vars, min_obs=60, max_na_pct=0.3)

print(f"\n АНАЛИЗ {len(region_quality_iqr)} РЕГИОНОВ (IQR)")
print(region_quality_iqr.to_string(index=False))

# СТАТИСТИКА
exclude_count = len(region_quality_iqr[region_quality_iqr['Status'] == 'EXCLUDE'])
keep_count = len(region_quality_iqr) - exclude_count

print(f"\n СТАТИСТИКА:")
print(f"Всего регионов: {len(region_quality_iqr)}")
print(f"Исключено: {exclude_count} ({exclude_count/len(region_quality_iqr)*100:.1f}%)")
print(f"Оставлено: {keep_count}")

# СПИСОК ХОРОШИХ РЕГИОНОВ
good_regions_iqr = region_quality_iqr[region_quality_iqr['Status'] == 'KEEP']['Region'].tolist()

# ФИЛЬТРАЦИЯ ДАННЫХ
df_reg_analys= df_reg_analys[df_reg_analys['Region'].isin(good_regions_iqr)].copy()
print(f"\n Форма после очистки (IQR): {df_reg_analys.shape}")




In [ ]:
###############################
# ОПИСАТЕЛЬНЫЕ СТАТИСТИКИ - ИСХОДНЫЕ ПАНЕЛЬНЫЕ ДАННЫЕ
###############################
descriptive_df = df_reg_analys[[
    'Int_Rate_FL',
    'Int_Rate_Mort',
    'Credit_impulse',
    'Int_Rate_ConsCred',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Cred_structure',
    'Def_Zadolg_Fl',
    'Def_Zadolg_Mort',
    'Def_Zadolg_ConsCred',
    #'New_Loans_Progr',
    'New_Loans_Fl',
    'New_Loans_Mort',
    'New_Loans_ConsCred',
    'Mon_Shock',
    'Exc_rate',
    'Inflation_Expectations',
    'Bonds_Rate_Correct_5Y'      
]]

desc = descriptive_df.describe().round(3).T

medians = descriptive_df.median()
desc['median'] = medians.round(3)

desc = desc.rename(columns={
    'mean': 'Среднее',
    'median': 'Медиана',
    'std': 'ст. откл.',
    'min': 'Мин.',
    'max': 'Макс.',
})

print("\n" + "="*60)
print("ОПИСАТЕЛЬНЫЕ СТАТИСТИКИ - ИСХОДНЫЕ ПАНЕЛЬНЫЕ ДАННЫЕ")
print("="*60)
print(desc[['Среднее', 'Медиана', 'ст. откл.', 'Мин.', 'Макс.']])

In [ ]:
###############################
# КОРРЕЛЯЦИОННАЯ МАТРИЦА - ИСХОДНЫЕ ПАНЕЛЬНЫЕ ДАННЫЕ
###############################
dd = descriptive_df.corr()
plt.figure(figsize=(12, 10))
fig = sns.heatmap(dd, annot=True,
                  fmt=".2f",
                  linewidth=0.5,
                  linecolor='white',
                  annot_kws={"size": 10},
                  cmap='coolwarm')

fig.set_title("Корреляционная матрица (начальные панельные данные)",
             fontsize=16, pad=20)

fig.set_xticklabels(fig.get_xticklabels(),
                   rotation=45,
                   ha='right',
                   fontsize=12)

plt.tight_layout()

In [ ]:
###############################
    #VIF-анализ - начальные панельные данные спецификация 1
###############################

vif_variables = [
    'Int_Rate_FL_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Fl',
    #'New_Loans_Progr',
    'Mon_Shock',
    'New_Loans_Fl',
    'Bonds_Rate_Correct_5Y',
    'Exc_rate',
    'Inflation_Expectations'
]
df_vif = df_reg_analys[vif_variables].copy()
df_vif = df_vif.dropna()

X_vif = sm.add_constant(df_vif[vif_variables])

# Расчет VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = vif_variables
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i+1) for i in range(len(vif_variables))]
vif_data = vif_data.sort_values('VIF', ascending=False)

print("\n" + "="*60)
print("РЕЗУЛЬТАТЫ VIF АНАЛИЗА - НАЧАЛЬНЫЕ ПАНЕЛЬНЫЕ ДАННЫЕ СПЕЦИФИКАЦИЯ 1")
print("="*60)
print(vif_data.to_string(index=False))

In [ ]:
###############################
    #VIF-анализ - начальные панельные данные спецификация 2
###############################

vif_variables = [
    'Int_Rate_ConsCred_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_ConsCred',
    #'New_Loans_Progr',
    'Mon_Shock',
    'New_Loans_ConsCred',
    'Bonds_Rate_Correct_5Y',
    'Exc_rate',
    'Inflation_Expectations'
]
df_vif = df_reg_analys[vif_variables].copy()
df_vif = df_vif.dropna()

X_vif = sm.add_constant(df_vif[vif_variables])

# Расчет VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = vif_variables
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i+1) for i in range(len(vif_variables))]
vif_data = vif_data.sort_values('VIF', ascending=False)

print("\n" + "="*60)
print("РЕЗУЛЬТАТЫ VIF АНАЛИЗА - НАЧАЛЬНЫЕ ПАНЕЛЬНЫЕ ДАННЫЕ СПЕЦИФИКАЦИЯ 2")
print("="*60)
print(vif_data.to_string(index=False))

In [ ]:
###############################
    #VIF-анализ - начальные панельные данные спецификация 3
###############################

vif_variables = [
    'Int_Rate_Mort_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Mort',
    #'New_Loans_Progr',
    'New_Loans_Mort',
    'Mon_Shock',
    'Bonds_Rate_Correct_5Y',
    'Exc_rate',
    'Inflation_Expectations'
]
 
df_vif = df_reg_analys[vif_variables].copy()
df_vif = df_vif.dropna()

X_vif = sm.add_constant(df_vif[vif_variables])

# Расчет VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = vif_variables
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i+1) for i in range(len(vif_variables))]
vif_data = vif_data.sort_values('VIF', ascending=False)

print("\n" + "="*60)
print("РЕЗУЛЬТАТЫ VIF АНАЛИЗА - НАЧАЛЬНЫЕ ПАНЕЛЬНЫЕ ДАННЫЕ СПЕЦИФИКАЦИЯ 3")
print("="*60)
print(vif_data.to_string(index=False))

In [ ]:
# ===== СЕЗОННАЯ КОРРЕКТИРОВКА ДЛЯ ПАНЕЛЬНЫХ ДАННЫХ =====


from statsmodels.tsa.x13 import x13_arima_analysis
import tempfile
import uuid

# Заглушаем предупреждения
warnings.filterwarnings('ignore')

# Путь к бинарнику X-13
X13_PATH = 'C:\\Program Files (x86)\\x13as'

# Проверка, что бинарник доступен
if not os.path.exists(X13_PATH):
    raise FileNotFoundError(f"X-13 binary not found at {X13_PATH}")

# Если X13_PATH — полный путь к бинарнику, берем его директорию для x12path
x13_dir = X13_PATH if os.path.isdir(X13_PATH) else os.path.dirname(X13_PATH)

# Создаем папку для логов
os.makedirs("x13_logs", exist_ok=True)

# ============ ОПРЕДЕЛЕНИЕ ПЕРЕМЕННЫХ ДЛЯ КОРРЕКТИРОВКИ ============

# ФЕДЕРАЛЬНЫЕ переменные (одна серия для всех регионов)
federal_variables = {
    'Exc_rate': 'Курс рубля', 
    'Inflation_Expectations': 'Инфляционные ожидания',
    'Bonds_Rate_Correct_5Y': 'Ставки на облигации скорректированные 5Y'
}

# РЕГИОНАЛЬНЫЕ переменные (серии по каждому региону)
regional_variables = {
    'Int_Rate_FL': 'Ставка на кредиты ФЛ',
    'Int_Rate_FL_lag1' : 'Ставка на кредиты ФЛ лаг',
    'Int_Rate_Mort': 'Ставка на ипотеку',
    'Int_Rate_Mort_lag1': 'Ставка на ипотеку лаг',
    'Credit_impulse': 'Кредитный импульс',
    'Int_Rate_ConsCred': 'Ставка на потребительский кредит',
    'Int_Rate_ConsCred_lag1': 'Ставка на потребительский кредит лаг',
    'Cred_nagr': 'Кредитная нагрузка',
    'D_top5_rozn': 'Доля топ-5 банков в розничном портфеле региона',
    'Fin_Dostup':'Кол-во пунктов банковского обслуживания на 100к чел.',
    'Cred_structure':'Доля ипотечных жилищных кредитов в розничном портфеле',
    'Def_Zadolg_Fl':'Доля просроченной задолженности по кредитам физических лиц',
    'Def_Zadolg_Mort':'Доля просроченной задолженности по ипотечным жилищным кредитам',
    'Def_Zadolg_ConsCred':'Доля просроченной задолженности по потребительским кредитам',
    'New_Loans_Fl': 'Новые кредиты ФЛ',
    'New_Loans_Mort': 'Новые кредиты ипотека',
    'New_Loans_ConsCred': 'Новые кредиты потребит.',
    
}
# ============ ФУНКЦИЯ ДЛЯ СЕЗОННОЙ КОРРЕКТИРОВКИ ОДНОЙ СЕРИИ ============

def seasonal_adjust_panel_series(series_data, variable_name, region_name=None, min_obs=24):
    """
    Выполнить X-13 сезонную корректировку для одной серии (панельные данные)
    
    Parameters:
    -----------
    series_data : pd.Series
        Временной ряд с DatetimeIndex
    variable_name : str
        Название переменной
    region_name : str, optional
        Название региона (для региональных переменных)
    min_obs : int
        Minimum number of observations
    
    Returns:
    --------
    dict с результатами или None при ошибке
    """
    try:
        # Проверка данных
        if len(series_data) < min_obs:
            return None
        
        # Убедиться, что индекс - DatetimeIndex
        if not isinstance(series_data.index, pd.DatetimeIndex):
            series_data.index = pd.to_datetime(series_data.index)
        
        # Установить частоту на месячную
        ts_regular = series_data.asfreq('MS').dropna()
        
        if len(ts_regular) < min_obs:
            return None
        
        # Информация о данных
        start_date = ts_regular.index[0]
        end_date = ts_regular.index[-1]
        
        # Создаем уникальную временную папку для каждого вызова X-13
        # чтобы избежать конфликтов файлов
        temp_dir = tempfile.mkdtemp(prefix=f"x13_{uuid.uuid4().hex[:8]}_")
        
        # Запуск X-13 с указанием уникальной временной директории
        try:
            res = x13_arima_analysis(
                endog=ts_regular,
                x12path=x13_dir,
                prefer_x13=True,
                outlier=True,
                trading=False,
                print_stdout=False
            )
            
            # Извлечение компонент
            adj = res.seasadj
            trend = res.trend
            seas = ts_regular - adj
            
        except Exception as e:
            # Если X-13 не сработал, попробуем без outlier
            try:
                res = x13_arima_analysis(
                    endog=ts_regular,
                    x12path=x13_dir,
                    prefer_x13=True,
                    outlier=False,  # Без определения выбросов
                    trading=False,
                    print_stdout=False
                )
                
                adj = res.seasadj
                trend = res.trend
                seas = ts_regular - adj
                
            except Exception as e2:
                # Если все еще не работает, пропускаем
                return None
        
        # Расчёт эффективности
        original_std = ts_regular.std()
        adj_std = adj.std()
        reduction = 100 * (1 - adj_std / original_std) if original_std > 0 else 0
        
        # Удаляем временную директорию
        try:
            import shutil
            shutil.rmtree(temp_dir, ignore_errors=True)
        except:
            pass
        
        return {
            'dates': ts_regular.index,
            'adj': adj.values,
            'trend': trend.values,
            'seasonal': seas.values,
            'reduction': reduction
        }
        
    except Exception as e:
        # Подавляем вывод ошибок
        return None

# ============ ОБРАБОТКА ФЕДЕРАЛЬНЫХ ПЕРЕМЕННЫХ ============

print("\n" + "="*70)
print("СЕЗОННАЯ КОРРЕКТИРОВКА ФЕДЕРАЛЬНЫХ ПЕРЕМЕННЫХ")
print("="*70)

federal_adjusted = {}

for col, name in federal_variables.items():
    if col not in df_reg_analys.columns:
        print(f"\n⚠️  {name} ({col}): столбец не найден. Пропуск.")
        continue
    
    print(f"\n{name} ({col}):")
    
    # Для федеральных переменных берем уникальные значения по датам
    fed_data = df_reg_analys[['Date', col]].copy().dropna()
    
    if len(fed_data) == 0:
        print(f"  ✗ Нет данных. Пропуск.")
        continue
    
    # Убедиться, что одно значение на дату
    fed_data = fed_data.drop_duplicates('Date')
    fed_data = fed_data.set_index('Date')[col].sort_index()
    
    # Сезонная корректировка
    result = seasonal_adjust_panel_series(fed_data, col, region_name=None)
    
    if result is not None:
        federal_adjusted[col] = result
        print(f"  ✓ Успешно скорректировано, снижение волатильности: {result['reduction']:.1f}%")
        
        # Добавляем результаты обратно в df_reg_analys
        for date, adj_val, trend_val, seas_val in zip(
            result['dates'], result['adj'], result['trend'], result['seasonal']
        ):
            mask = df_reg_analys['Date'] == date
            df_reg_analys.loc[mask, f'{col}_adj'] = adj_val
            df_reg_analys.loc[mask, f'{col}_trend'] = trend_val
            df_reg_analys.loc[mask, f'{col}_seasonal'] = seas_val
    else:
        print(f"  ✗ Не удалось скорректировать")

# ============ ОБРАБОТКА РЕГИОНАЛЬНЫХ ПЕРЕМЕННЫХ ============

print("\n" + "="*70)
print("СЕЗОННАЯ КОРРЕКТИРОВКА РЕГИОНАЛЬНЫХ ПЕРЕМЕННЫХ")
print("="*70)

regional_adjusted = {}

for col, name in regional_variables.items():
    if col not in df_reg_analys.columns:
        print(f"\n⚠️  {name} ({col}): столбец не найден. Пропуск.")
        continue
    
    print(f"\n{name} ({col}):")
    
    # Получаем список регионов
    regions = df_reg_analys['Region'].unique()
    print(f"  Обрабатываю {len(regions)} регионов...")
    
    regional_adjusted[col] = {}
    successful_regions = 0
    
    for region in regions:
        # Фильтруем данные по региону
        region_data = df_reg_analys[df_reg_analys['Region'] == region][['Date', col]].copy().dropna()
        
        if len(region_data) < 24:
            continue  # Пропускаем регионы с недостаточным количеством данных
        
        # Подготавливаем временной ряд
        region_series = region_data.set_index('Date')[col].sort_index()
        
        # Сезонная корректировка
        result = seasonal_adjust_panel_series(region_series, col, region_name=region, min_obs=24)
        
        if result is not None:
            regional_adjusted[col][region] = result
            successful_regions += 1
            
            # Добавляем результаты в df_reg_analys
            for date, adj_val, trend_val, seas_val in zip(
                result['dates'], result['adj'], result['trend'], result['seasonal']
            ):
                mask = (df_reg_analys['Region'] == region) & (df_reg_analys['Date'] == date)
                if mask.any():
                    df_reg_analys.loc[mask, f'{col}_adj'] = adj_val
                    df_reg_analys.loc[mask, f'{col}_trend'] = trend_val
                    df_reg_analys.loc[mask, f'{col}_seasonal'] = seas_val
    
    print(f"  ✓ Успешно скорректировано {successful_regions}/{len(regions)} регионов")

# ============ ПРОВЕРКА ДОБАВЛЕННЫХ СТОЛБЦОВ ============

print("\n" + "="*70)
print("ПРОВЕРКА ДОБАВЛЕННЫХ СТОЛБЦОВ")
print("="*70)

all_variables = list(regional_variables.keys()) + list(federal_variables.keys())
added_columns = []

for col in all_variables:
    for suffix in ['_adj', '_trend', '_seasonal']:
        new_col = f'{col}{suffix}'
        if new_col in df_reg_analys.columns:
            added_columns.append(new_col)
            non_na_count = df_reg_analys[new_col].notna().sum()
            print(f"✓ {new_col}: {non_na_count} не-NaN значений")

print(f"\n✓ Всего добавлено столбцов: {len(added_columns)}")

# ============ СВОДКА РЕЗУЛЬТАТОВ ============

print("\n" + "="*70)
print("СВОДКА РЕЗУЛЬТАТОВ СЕЗОННОЙ КОРРЕКТИРОВКИ")
print("="*70)

# Федеральные переменные
print("\nФедеральные переменные:")
print("-" * 50)
print(f"{'Переменная':<30} {'Волатильность ↓':<15}")
print("-" * 50)

for col in federal_variables:
    adj_col = f'{col}_adj'
    if adj_col in df_reg_analys.columns and col in federal_adjusted:
        reduction = federal_adjusted[col]['reduction']
        print(f"{federal_variables[col]:<30} {reduction:>14.1f}%")

# Региональные переменные (среднее по регионам)
print("\n\nРегиональные переменные (среднее по регионам):")
print("-" * 50)
print(f"{'Переменная':<30} {'Волатильность ↓':<15}")
print("-" * 50)

for col in regional_variables:
    adj_col = f'{col}_adj'
    if adj_col in df_reg_analys.columns and col in regional_adjusted:
        # Рассчитываем среднее снижение волатильности
        reductions = []
        for region_data in regional_adjusted[col].values():
            reductions.append(region_data['reduction'])
        
        if reductions:
            avg_reduction = np.mean(reductions)
            print(f"{regional_variables[col]:<30} {avg_reduction:>14.1f}%")

# ============ ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ ============

print("\n" + "="*70)
print("ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ (ПРИМЕРЫ)")
print("="*70)

# Выбираем несколько регионов для визуализации
sample_regions = df_reg_analys['Region'].unique()[:3]  # Первые 3 региона

# Визуализация для нескольких переменных
for col in list(regional_variables.keys())[:2]:  # Первые 2 региональные переменные
    adj_col = f'{col}_adj'
    
    if adj_col not in df_reg_analys.columns:
        continue
    
    for region in sample_regions:
        # Фильтруем данные для региона
        region_mask = (df_reg_analys['Region'] == region) & df_reg_analys[adj_col].notna()
        region_data = df_reg_analys[region_mask]
        
        if len(region_data) == 0:
            continue
        
        # Создаем график
        fig, axes = plt.subplots(2, 2, figsize=(14, 8))
        
        # График 1: Исходный vs скорректированный
        ax = axes[0, 0]
        dates = region_data['Date']
        ax.plot(dates, region_data[col], label='Исходный', alpha=0.6, linewidth=1)
        ax.plot(dates, region_data[adj_col], label='Сезонно скорр.', linewidth=1.5)
        ax.set_title(f'{regional_variables[col]}\nРегион: {region}')
        ax.legend()
        ax.grid(alpha=0.3)
        ax.tick_params(axis='x', rotation=45)
        
        # График 2: Сезонная компонента
        ax = axes[0, 1]
        seasonal_col = f'{col}_seasonal'
        if seasonal_col in region_data.columns:
            ax.plot(dates, region_data[seasonal_col], color='red', linewidth=1.5)
            ax.axhline(0, color='black', linestyle='--', alpha=0.3)
            ax.set_title('Сезонная компонента')
            ax.grid(alpha=0.3)
            ax.tick_params(axis='x', rotation=45)
        
        # График 3: Тренд
        ax = axes[1, 0]
        trend_col = f'{col}_trend'
        if trend_col in region_data.columns:
            ax.plot(dates, region_data[trend_col], color='green', linewidth=1.5)
            ax.set_title('Тренд')
            ax.grid(alpha=0.3)
            ax.tick_params(axis='x', rotation=45)
        
        # График 4: Статистика
        ax = axes[1, 1]
        ax.axis('off')
        
        # Расчет статистик
        if len(region_data) > 0:
            original_std = region_data[col].std()
            adj_std = region_data[adj_col].std()
            reduction = 100 * (1 - adj_std / original_std) if original_std > 0 else 0
            
            stats_text = f"Статистика для {region}:\n\n"
            stats_text += f"Период: {region_data['Date'].min().strftime('%Y-%m')} - {region_data['Date'].max().strftime('%Y-%m')}\n"
            stats_text += f"Наблюдений: {len(region_data)}\n"
            stats_text += f"Снижение волатильности: {reduction:.1f}%\n"
            stats_text += f"Стд. откл. до: {original_std:.4f}\n"
            stats_text += f"Стд. откл. после: {adj_std:.4f}"
            
            ax.text(0.1, 0.5, stats_text, fontsize=10, va='center', linespacing=1.5)
        
        plt.tight_layout()
        
        # Сохраняем график
        region_safe = region.replace('/', '_').replace('\\', '_').replace(':', '_')
        filename = f'x13_logs/panel_{col}_{region_safe}_seasonal.png'
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()

# Визуализация для федеральных переменных
print("\nГрафики федеральных переменных...")

for col, name in federal_variables.items():
    adj_col = f'{col}_adj'
    
    if adj_col not in df_reg_analys.columns:
        continue
    
    # Фильтруем данные
    fed_mask = df_reg_analys[adj_col].notna()
    fed_data = df_reg_analys[fed_mask].drop_duplicates('Date')
    
    if len(fed_data) == 0:
        continue
    
    # Создаем график
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    
    # График 1: Исходный vs скорректированный
    ax = axes[0, 0]
    dates = fed_data['Date']
    ax.plot(dates, fed_data[col], label='Исходный', alpha=0.6, linewidth=1)
    ax.plot(dates, fed_data[adj_col], label='Сезонно скорр.', linewidth=1.5)
    ax.set_title(f'{name}\nФедеральная переменная')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.tick_params(axis='x', rotation=45)
    
    # График 2: Сезонная компонента
    ax = axes[0, 1]
    seasonal_col = f'{col}_seasonal'
    if seasonal_col in fed_data.columns:
        ax.plot(dates, fed_data[seasonal_col], color='red', linewidth=1.5)
        ax.axhline(0, color='black', linestyle='--', alpha=0.3)
        ax.set_title('Сезонная компонента')
        ax.grid(alpha=0.3)
        ax.tick_params(axis='x', rotation=45)
    
    # График 3: Тренд
    ax = axes[1, 0]
    trend_col = f'{col}_trend'
    if trend_col in fed_data.columns:
        ax.plot(dates, fed_data[trend_col], color='green', linewidth=1.5)
        ax.set_title('Тренд')
        ax.grid(alpha=0.3)
        ax.tick_params(axis='x', rotation=45)
    
    # График 4: Статистика
    ax = axes[1, 1]
    ax.axis('off')
    
    # Расчет статистик
    if len(fed_data) > 0 and col in federal_adjusted:
        reduction = federal_adjusted[col]['reduction']
        original_std = fed_data[col].std()
        adj_std = fed_data[adj_col].std()
        
        stats_text = f"Статистика для {name}:\n\n"
        stats_text += f"Период: {fed_data['Date'].min().strftime('%Y-%m')} - {fed_data['Date'].max().strftime('%Y-%m')}\n"
        stats_text += f"Наблюдений: {len(fed_data)}\n"
        stats_text += f"Снижение волатильности: {reduction:.1f}%\n"
        stats_text += f"Стд. откл. до: {original_std:.4f}\n"
        stats_text += f"Стд. откл. после: {adj_std:.4f}"
        
        ax.text(0.1, 0.5, stats_text, fontsize=10, va='center', linespacing=1.5)
    
    plt.tight_layout()
    
    # Сохраняем график
    filename = f'x13_logs/federal_{col}_seasonal.png'
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

# ============ ФИНАЛЬНАЯ СТАТИСТИКА ============

print("\n" + "="*70)
print("ФИНАЛЬНАЯ СТАТИСТИКА")
print("="*70)

print(f"\nОбработано федеральных переменных: {len(federal_adjusted)}/{len(federal_variables)}")
print(f"Обработано региональных переменных: {len(regional_adjusted)}/{len(regional_variables)}")
print(f"Общее количество добавленных столбцов: {len(added_columns)}")
print(f"Форма df_reg_analys: {df_reg_analys.shape}")

# Проверяем наличие столбцов с суффиксами
adj_columns = [col for col in df_reg_analys.columns if col.endswith('_adj')]
trend_columns = [col for col in df_reg_analys.columns if col.endswith('_trend')]
seasonal_columns = [col for col in df_reg_analys.columns if col.endswith('_seasonal')]

print(f"\nСтолбцов _adj: {len(adj_columns)}")
print(f"Столбцов _trend: {len(trend_columns)}")
print(f"Столбцов _seasonal: {len(seasonal_columns)}")

print("\n✓ Скорректированные ряды добавлены в df_reg_analys")
print("✓ Столбцы с суффиксами: _adj (скорр.), _trend (тренд), _seasonal (сезон)")

# Сортируем по дате и региону
df_reg_analys = df_reg_analys.sort_values(['Region', 'Date']).reset_index(drop=True)

In [ ]:
df_reg_analys.info()

In [ ]:
# ===== УДАЛЕНИЕ НЕНУЖНЫХ ДАННЫХ =====

print("="*70)
print("УДАЛЕНИЕ НЕНУЖНЫХ ДАННЫХ")
print("="*70)

print(f"\n ИСХОДНЫЕ ДАННЫЕ:")
print(f"Форма датасета: {df_reg_analys.shape}")
print(f"Всего столбцов: {len(df_reg_analys.columns)}")

# Определяем столбцы, которые оставляем:
# 1. 'Date' и 'Region' (служебные)
# 2. Все столбцы, заканчивающиеся на '_adj' 

columns_to_keep = ['Date', 'Region', 'Mon_Shock']
adjusted_columns = [col for col in df_reg_analys.columns 
                   if col.endswith('_adj')]

columns_to_keep.extend(adjusted_columns)

print(f"\n СТОЛБЦЫ ДЛЯ СОХРАНЕНИЯ:")
print(f"Служебные: {['Date', 'Region']}")
print(f"+ {['Mon_Shock']}")
print(f"Adjusted (_adj): {len([c for c in adjusted_columns if c.endswith('_adj')])} столбцов")

# Определяем столбцы на удаление
columns_to_drop = [col for col in df_reg_analys.columns if col not in columns_to_keep]

print(f"\n СТОЛБЦЫ НА УДАЛЕНИЕ:")
print(f"Всего на удаление: {len(columns_to_drop)}")
for i, col in enumerate(columns_to_drop, 1):
    print(f"  {i}. {col}")

# Удаляем столбцы
df_reg_analys = df_reg_analys.drop(columns=columns_to_drop)

print(f"\n" + "="*70)
print("ПОСЛЕ УДАЛЕНИЯ:")
print("="*70)
print(f"✓ Форма датасета: {df_reg_analys.shape}")
print(f"✓ Всего столбцов: {len(df_reg_analys.columns)}")

# Статистика по столбцам
adj_count = len([c for c in df_reg_analys.columns if c.endswith('_adj')])
trend_count = len([c for c in df_reg_analys.columns if c.endswith('_trend')])
seasonal_count = len([c for c in df_reg_analys.columns if c.endswith('_seasonal')])

In [ ]:
###########################################################
# Вспомогательные функции: KPSS, Hadri LM (panel), Levin–Lin–Chu (panel)
###########################################################

def _kpss_level_stat(y):
    """KPSS статистика для уровня стационарности (без тренда), гомоскедастичная долгосрочная дисперсия."""
    y = np.asarray(y, float)
    T = len(y)
    u = y - y.mean()
    S = np.cumsum(u)
    sigma2 = np.mean(u**2)
    return np.sum(S**2) / (T**2 * sigma2)

def hadri_panel_test(df, id_col='Region', time_col='Date', y_col='y', trend=True):
    """
    Упрощённый тест панельной стационарности Hadri (2000):
    - Строит KPSS-подобные статистики по каждому юниту и усредняет их.
    - Возвращает среднюю статистику и эмпирическое p-значение по бутстрепу.
    H0: все панели стационарны.
    """
    d = df[[id_col, time_col, y_col]].dropna().copy()
    d = d.sort_values([id_col, time_col])

    # Требуется сбалансированная панель для этой реализации
    counts = d.groupby(id_col)[y_col].size().values
    if not np.all(counts == counts[0]):
        raise ValueError("Тест Hadri требует сбалансированную панель")

    kpss_vals = []

    for _, g in d.groupby(id_col):
        y = g[y_col].values.astype(float)

        if trend:
            # Очистка от тренда: константа + линейное время
            t = np.arange(len(y))
            X = sm.add_constant(t)
            res = sm.OLS(y, X).fit()
            u = res.resid
            S = np.cumsum(u)
            sigma2 = np.mean(u**2)
            T = len(y)
            stat = np.sum(S**2) / (T**2 * sigma2)
        else:
            stat = _kpss_level_stat(y)

        kpss_vals.append(stat)

    kpss_vals = np.asarray(kpss_vals, float)
    N = len(kpss_vals)

    # Средняя KPSS-статистика (Hadri LM)
    lm_mean = float(kpss_vals.mean())

    # Эмпирическое p-значение через бутстреп перестановок по юнитам
    B = 1000
    if N > 1:
        rng = np.random.default_rng(12345)
        lm_boot = []
        for _ in range(B):
            idx = rng.integers(0, N, size=N)
            lm_boot.append(kpss_vals[idx].mean())
        lm_boot = np.asarray(lm_boot)
        pvalue = float((lm_boot >= lm_mean).mean())
    else:
        pvalue = np.nan

    return {
        "statistic": lm_mean,
        "pvalue": pvalue,
        "kpss_mean": lm_mean,
        "kpss_std": float(kpss_vals.std(ddof=1)) if N > 1 else np.nan,
        "N": int(N)
    }

def levin_lin_chu_test(df, id_col='Region', time_col='Date', y_col='y',
                       lags=1, trend=True):
    """
    Упрощённая реализация Levin–Lin–Chu:
    - выравнивание по общей временной сетке,
    - отбрасывание несбалансированных юнитов после лагов.
    H0: присутствует единичный корень (панели НЕ стационарны).
    """
    d = df[[id_col, time_col, y_col]].copy()
    d = d.dropna(subset=[id_col, time_col, y_col])

    # Общая временная сетка
    all_times = np.sort(d[time_col].unique())

    dy_list = []
    ylag_list = []
    lagdiff_list = []
    trend_list = []
    fe_ids = []

    unit_idx = 0

    for _, g in d.groupby(id_col):
        # Выравниваем по общей сетке времени
        g = g.set_index(time_col).reindex(all_times).sort_index()
        y = pd.to_numeric(g[y_col], errors='coerce').values

        # Если все пропуски — пропускаем юнит
        if np.isnan(y).all():
            continue

        # Убираем ведущие и хвостовые NaN блоком
        mask_valid = ~np.isnan(y)
        if mask_valid.sum() <= (lags + 2):
            continue
        first = np.argmax(mask_valid)
        last = len(mask_valid) - np.argmax(mask_valid[::-1])
        y = y[first:last]

        # После обрезки снова проверка длины
        if len(y) <= (lags + 2):
            continue

        # Строим Δy и лаги
        dy = np.diff(y)
        ylag = y[:-1]

        # Лаги разностей
        lag_diffs = []
        for k in range(1, lags + 1):
            lag_k = np.r_[ [np.nan]*k, np.diff(y)[:-k] ]
            lag_diffs.append(lag_k[1:])
        if lags > 0:
            lag_diffs = np.column_stack(lag_diffs)
        else:
            lag_diffs = np.empty((len(dy), 0))

        T_eff = len(dy)
        ylag = ylag[1:]
        if len(ylag) != T_eff:
            T_eff = min(T_eff, len(ylag))
        dy = dy[:T_eff]
        ylag = ylag[:T_eff]
        lag_diffs = lag_diffs[:T_eff, :]

        if T_eff <= 0:
            continue

        dy_list.append(dy)
        ylag_list.append(ylag)
        lagdiff_list.append(lag_diffs)

        if trend:
            t = np.arange(1, T_eff + 1)
            trend_list.append(t)

        fe_ids.append(np.repeat(unit_idx, T_eff))
        unit_idx += 1

    # Если после отсева юнитов ничего не осталось
    if len(dy_list) == 0:
        raise ValueError("LLC: нет полезных панелей после балансировки; проверьте пропуски и лаги")

    dy_all = np.concatenate(dy_list)
    ylag_all = np.concatenate(ylag_list)
    if lags > 0:
        lagdiff_all = np.vstack(lagdiff_list)
    else:
        lagdiff_all = np.empty((len(dy_all), 0))

    ids_all = np.concatenate(fe_ids)
    fe_dummies = pd.get_dummies(ids_all, drop_first=True).values

    if trend:
        trend_all = np.concatenate(trend_list)[:, None]
    else:
        trend_all = np.empty((len(dy_all), 0))

    # Финальная проверка на одинаковую длину
    n_obs = len(dy_all)
    ylag_all = ylag_all[:n_obs]
    lagdiff_all = lagdiff_all[:n_obs, :]
    fe_dummies = fe_dummies[:n_obs, :]
    trend_all = trend_all[:n_obs, :]

    X = np.column_stack([ylag_all[:, None], lagdiff_all, fe_dummies, trend_all])
    X = sm.add_constant(X, has_constant="add")

    model = sm.OLS(dy_all, X)
    res = model.fit()

    rho_idx = 1  # const на позиции 0, rho на 1
    rho_hat = res.params[rho_idx]
    t_stat = res.tvalues[rho_idx]
    pvalue = 2 * (1 - stats.norm.cdf(abs(t_stat)))

    return {
        "rho_hat": float(rho_hat),
        "t_stat": float(t_stat),
        "pvalue": float(pvalue),
        "N": int(unit_idx)
    }


###########################################################
# Основная функция для тестирования единичных корней
###########################################################

def run_panel_unit_root_tests_python(df_reg_analys: pd.DataFrame,
                                     panel_vars: list = None) -> pd.DataFrame:
    """
    Применяет тесты на единичные корни к переменным из списка.

    Parameters
    ----------
    df_reg_analys : pd.DataFrame
        Исходный датасет с колонками 'Region', 'Date' и тестируемыми переменными.

    panel_vars : list, optional
        Список переменных для тестирования.
        Если None, тестируются все переменные с суффиксом '_adj'.
    """

    # === ПОЛУЧЕНИЕ ПЕРЕМЕННЫХ ДЛЯ ТЕСТИРОВАНИЯ ===
    if panel_vars is None:
        panel_vars = [
            col for col in df_reg_analys.columns
            if col.endswith('_adj') and col not in ['Date', 'Region']
        ]
    else:
        missing_vars = [v for v in panel_vars if v not in df_reg_analys.columns]
        if missing_vars:
            print(f"⚠ Предупреждение: следующие переменные не найдены в датасете: {missing_vars}")
            panel_vars = [v for v in panel_vars if v in df_reg_analys.columns]

        if not panel_vars:
            raise ValueError("Нет переменных для тестирования")

    print(f"Тестирование {len(panel_vars)} переменных на единичные корни")
    print("=" * 70)
    print(f"Переменные для тестирования:")
    for i, var in enumerate(panel_vars, 1):
        print(f"  {i}. {var}")
    print("=" * 70)

    results_rows = []
    pdf = df_reg_analys.copy()

    for var in panel_vars:
        try:
            x = pdf[var].values
            x_clean = x[~pd.isna(x)]
            if len(x_clean) <= 10:
                print(f"⚠ {var}: недостаточно данных (< 10 наблюдений)")
                continue

            # === Hadri ===
            hadri_stat = np.nan
            hadri_pval = np.nan
            try:
                hadri_res = hadri_panel_test(
                    df_reg_analys[['Region', 'Date', var]].rename(columns={var: 'y'}),
                    id_col='Region',
                    time_col='Date',
                    y_col='y',
                    trend=True
                )
                hadri_stat = float(hadri_res["statistic"])
                hadri_pval = float(hadri_res["pvalue"])
            except Exception as e:
                print(f"Ошибка теста Hadri для {var}: {e}")

            # === Levin–Lin–Chu ===
            llc_stat = np.nan
            llc_pval = np.nan
            try:
                llc_res = levin_lin_chu_test(
                    df_reg_analys[['Region', 'Date', var]].rename(columns={var: 'y'}),
                    id_col='Region',
                    time_col='Date',
                    y_col='y',
                    lags=1,
                    trend=True
                )
                llc_stat = float(llc_res["t_stat"])
                llc_pval = float(llc_res["pvalue"])
            except Exception as e:
                print(f"Ошибка теста Levin–Lin–Chu для {var}: {e}")

            # === PP и DF-GLS по регионам ===
            pp_stats = []
            pp_pvalues = []
            pp_rejections = 0

            dfgls_stats = []
            dfgls_rejections = 0

            regions = pdf['Region'].dropna().unique()

            for region in regions:
                region_series = pdf.loc[pdf['Region'] == region, var]
                region_data = pd.to_numeric(region_series, errors='coerce').dropna().values

                if len(region_data) > 5:
                    # PP (приблизительно через ADF)
                    try:
                        adf_res = sm.tsa.stattools.adfuller(region_data, regression='c', autolag='AIC')
                        pp_stat = adf_res[0]
                        pp_pval = adf_res[1]
                        pp_stats.append(pp_stat)
                        pp_pvalues.append(pp_pval)
                        if pp_pval < 0.05:
                            pp_rejections += 1
                    except Exception:
                        pass

                    # DF-GLS (через GLS-детрендинг + ADF)
                    try:
                        T = len(region_data)
                        y = region_data
                        alpha = 1 - 7 / T
                        y_tilde = y[1:] - alpha * y[:-1]
                        X = np.ones(T)
                        X_tilde = X[1:] - alpha * X[:-1]
                        beta = np.linalg.lstsq(X_tilde[:, None], y_tilde, rcond=None)[0][0]
                        u = y - beta * X
                        du = np.diff(u)
                        u_lag = u[:-1]
                        adf_gls = sm.OLS(du, sm.add_constant(u_lag)).fit()
                        stat = adf_gls.tvalues[1]
                        dfgls_stats.append(stat)
                        if stat < -2.86:
                            dfgls_rejections += 1
                    except Exception:
                        pass

            # Агрегация PP
            pp_mean = np.nan
            pp_median = np.nan
            pp_rejection_rate = np.nan
            if len(pp_stats) > 0:
                pp_mean = float(np.nanmean(pp_stats))
                pp_median = float(np.nanmedian(pp_stats))
                pp_rejection_rate = float(pp_rejections / len(pp_stats) * 100.0)

            # Агрегация DF-GLS
            dfgls_mean = np.nan
            dfgls_median = np.nan
            dfgls_rejection_rate = np.nan
            if len(dfgls_stats) > 0:
                dfgls_mean = float(np.nanmean(dfgls_stats))
                dfgls_median = float(np.nanmedian(dfgls_stats))
                dfgls_rejection_rate = float(dfgls_rejections / len(dfgls_stats) * 100.0)

            results_rows.append({
                "Переменная": var,
                "Hadri_Stat": hadri_stat,
                "Hadri_PValue": hadri_pval,
                "LLC_Stat": llc_stat,
                "LLC_PValue": llc_pval,
                "PP_Mean": pp_mean,
                "PP_Median": pp_median,
                "PP_Rejections_%": pp_rejection_rate,
                "DFGLS_Mean": dfgls_mean,
                "DFGLS_Median": dfgls_median,
                "DFGLS_Rejections_%": dfgls_rejection_rate
            })

        except Exception as e:
            print(f"Общая ошибка для {var}: {e}")

    results = pd.DataFrame(results_rows)

    print("\n" + "=" * 70)
    print("РЕЗУЛЬТАТЫ ТЕСТОВ НА ЕДИНИЧНЫЕ КОРНИ")
    print("=" * 70)

    # === ИНТЕРПРЕТАЦИЯ РЕЗУЛЬТАТОВ ===
    results['Hadri_Stationarity'] = results['Hadri_PValue'] >= 0.05
    results['LLC_Stationarity'] = results['LLC_PValue'] < 0.05
    results['PP_Stationarity'] = results['PP_Rejections_%'] >= 50
    results['DFGLS_Stationarity'] = results['DFGLS_Rejections_%'] >= 50

    results['Overall_Stationarity'] = (
        (results['Hadri_Stationarity'] & results['LLC_Stationarity']) &
        (results['PP_Stationarity'] | results['DFGLS_Stationarity'])
    )

    print(results.to_string(index=False))

    print("\n" + "=" * 70)
    print("ИТОГОВАЯ СВОДКА")
    print("=" * 70)
    print(f"Всего переменных протестировано: {len(results)}")
    print(f"Тест Hadri (H0: стационарность): {results['Hadri_Stationarity'].sum()} стационарны")
    print(f"Тест Levin–Lin–Chu (отвергаем H0: единичный корень): {results['LLC_Stationarity'].sum()} стационарны")
    print(f"Тест PP: {results['PP_Stationarity'].sum()} стационарны (≥50% регионов)")
    print(f"Тест DF-GLS: {results['DFGLS_Stationarity'].sum()} стационарны (≥50% регионов)")
    print(f"\n✓ Общий вывод (Hadri И LLC + минимум 1 из PP/DF-GLS): {results['Overall_Stationarity'].sum()} стационарны")

    # Вывод нестационарных переменных
    non_stationary_vars = results[~results['Overall_Stationarity']]['Переменная'].tolist()
    if non_stationary_vars:
        print(f"\n  Нестационарные переменные: {len(non_stationary_vars)}")
        for var in non_stationary_vars:
            print(f"  - {var}")

    return results


# ===== ЗАПУСК ТЕСТОВ НА КОНКРЕТНЫЕ ПЕРЕМЕННЫЕ =====

print("\n" + "="*70)
print("ЗАПУСК ТЕСТОВ НА ЕДИНИЧНЫЕ КОРНИ ДЛЯ КОНКРЕТНЫХ ПЕРЕМЕННЫХ")
print("="*70)

# === СПИСОК ПЕРЕМЕННЫХ ДЛЯ ТЕСТИРОВАНИЯ ===
custom_vars = [
    'Int_Rate_FL_adj',
    'Int_Rate_FL_lag1_adj',
    'Int_Rate_Mort_adj',
    'Credit_impulse_adj',
    'Int_Rate_Mort_lag1_adj',
    'Int_Rate_ConsCred_adj',
    'Int_Rate_ConsCred_lag1_adj',
    'Cred_nagr_adj',
    'D_top5_rozn_adj',
    'Fin_Dostup_adj',
    'Cred_structure_adj',
    'Def_Zadolg_Fl_adj',
    'Def_Zadolg_Mort_adj',
    'Def_Zadolg_ConsCred_adj',
    'New_Loans_Fl_adj',
    'New_Loans_Mort_adj',
    'New_Loans_ConsCred_adj',
    'Inflation_Expectations_adj',
    'Bonds_Rate_Correct_5Y_adj',
    'Exc_rate_adj'
]

# Запуск функции с пользовательским списком переменных
results_seasonal_adjusted = run_panel_unit_root_tests_python(df_reg_analys, panel_vars=custom_vars)

In [ ]:
df_reg_analys.info()

In [ ]:
###############################
# Переход к первым разностям панельных данных
# С логарифмированием переменных выдач
###############################
df_reg = df_reg_analys.copy()

if isinstance(df_reg.index, pd.MultiIndex):
    df_reg = df_reg.reset_index()

# ============ ЛОГАРИФМИРОВАНИЕ ВЫДАЧ ============

loan_vars = ['New_Loans_Fl_adj', 'New_Loans_Mort_adj', 'New_Loans_ConsCred_adj']

for var in loan_vars:
    if var in df_reg.columns:
        df_reg[f'ln_{var}'] = np.log(df_reg[var])

# ============ СОЗДАНИЕ ПЕРВЫХ РАЗНОСТЕЙ ТОЛЬКО ДЛЯ НЕСТАЦИОНАРНЫХ ПЕРЕМЕННЫХ ============

exclude_cols = ['Region', 'Date']

# список скорректированных переменных, для которых тесты показали нестационарность
nonstat_vars = results_seasonal_adjusted.loc[
    results_seasonal_adjusted['Overall_Stationarity'] == False,  # нестационарные
    'Переменная'].tolist()

# защитимся от случая, если в results есть переменные, которых уже нет в df_reg
nonstat_vars = [v for v in nonstat_vars if v in df_reg.columns]

# логарифмы выдач
ln_loan_vars = [f'ln_{var}' for var in loan_vars if f'ln_{var}' in df_reg.columns]

# обычные переменные, для которых нужно строить разности:
#  - только нестационарные
#  - исключаем служебные, логарифмы и уже существующие d_
initial_vars = [
    col for col in nonstat_vars
    if col not in exclude_cols
    and not col.startswith('d_')
    and not col.startswith('ln_')
]

# Разности для обычных нестационарных переменных
for var in initial_vars:
    df_reg[f'd_{var}'] = df_reg.groupby('Region')[var].diff()

# Разности для логарифмов выдач (их всегда дифференцируем, как раньше)
for var in ln_loan_vars:
    d_var = f'd_{var}'
    df_reg[d_var] = df_reg.groupby('Region')[var].diff()

# ============ ПРОВЕРКА СОЗДАННЫХ СТОЛБЦОВ ============

d_cols = [col for col in df_reg.columns if col.startswith('d_')]

exog_vars = [
    'd_Int_Rate_FL_adj',
    'd_Int_Rate_FL_lag1_adj',
    'd_Int_Rate_Mort_adj',
    'Credit_impulse_adj',
    'd_Int_Rate_Mort_lag1_adj',
    'd_Int_Rate_ConsCred_adj',
    'd_Int_Rate_ConsCred_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_Fl_adj',
    'd_Def_Zadolg_Mort_adj',
    'd_Def_Zadolg_ConsCred_adj',
    'd_ln_New_Loans_Fl_adj',         
    'd_ln_New_Loans_Mort_adj',       
    'd_ln_New_Loans_ConsCred_adj',   
    'Inflation_Expectations_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'Mon_Shock',
    'd_Exc_rate_adj'
]

# Проверяем, какие переменные реально существуют
exog_vars_existing = [var for var in exog_vars if var in df_reg.columns]
exog_vars_missing = [var for var in exog_vars if var not in df_reg.columns]

df_exog = df_reg[['Region', 'Date'] + exog_vars_existing].copy()
df_exog = df_exog.dropna()


In [ ]:
df_exog.head()

In [ ]:
df_exog.info()

In [ ]:

###############################
# ОПИСАТЕЛЬНЫЕ СТАТИСТИКИ - ПАНЕЛЬНЫЕ ДАННЫЕ В ПЕРВЫХ РАЗНОСТЯХ
###############################
descriptive_df = df_exog[[
    'd_Int_Rate_FL_adj',
    'd_Int_Rate_Mort_adj',
    'Credit_impulse_adj',
    'd_Int_Rate_ConsCred_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_Fl_adj',
    'd_Def_Zadolg_Mort_adj',
    'd_Def_Zadolg_ConsCred_adj',
    'd_ln_New_Loans_Fl_adj',         
    'd_ln_New_Loans_Mort_adj',       
    'd_ln_New_Loans_ConsCred_adj',   
    'Inflation_Expectations_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'Mon_Shock',
    'd_Exc_rate_adj'
]]

desc = descriptive_df.describe().round(3).T

medians = descriptive_df.median()
desc['median'] = medians.round(3)

desc = desc.rename(columns={
    'mean': 'Среднее',
    'median': 'Медиана',
    'std': 'ст. откл.',
    'min': 'Мин.',
    'max': 'Макс.',
})

print("\n" + "="*60)
print("ОПИСАТЕЛЬНЫЕ СТАТИСТИКИ - ПАНЕЛЬНЫЕ ДАННЫЕ В ПЕРВЫХ РАЗНОСТЯХ")
print("="*60)
print(desc[['Среднее', 'Медиана', 'ст. откл.', 'Мин.', 'Макс.']])

In [ ]:
###############################
# КОРРЕЛЯЦИОННАЯ МАТРИЦА - ПАНЕЛЬНЫЕ ДАННЫЕ В ПЕРВЫХ РАЗНОСТЯХ
###############################
dd = descriptive_df.corr()
plt.figure(figsize=(12, 10))
fig = sns.heatmap(dd, annot=True,
                  fmt=".2f",
                  linewidth=0.5,
                  linecolor='white',
                  annot_kws={"size": 10},
                  cmap='coolwarm')

fig.set_title("Корреляционная матрица (панельные данные в первых разностях)",
             fontsize=16, pad=20)

fig.set_xticklabels(fig.get_xticklabels(),
                   rotation=45,
                   ha='right',
                   fontsize=12)

plt.tight_layout()

In [ ]:
###############################
    #VIF-анализ - панельные данные спецификация в первых разностях 1 спецификация
###############################

vif_variables = [
    'd_Int_Rate_FL_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_Fl_adj',
   # 'd_ln_New_Loans_Progr',
    'Mon_Shock',
    'd_ln_New_Loans_Fl_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_Exc_rate_adj',
    'Inflation_Expectations_adj'                   
]

df_vif = df_exog[vif_variables].copy()
df_vif = df_vif.dropna()
X_vif = sm.add_constant(df_vif[vif_variables])

# Расчет VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = vif_variables
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i+1) for i in range(len(vif_variables))]
vif_data = vif_data.sort_values('VIF', ascending=False)

print("\n" + "="*60)
print("РЕЗУЛЬТАТЫ VIF АНАЛИЗА - ПАНЕЛЬНЫЕ ДАННЫЕ В ПЕРВЫХ РАЗНОСТЯХ СПЕЦИФИКАЦИЯ 1")
print("="*60) 
print(vif_data.to_string(index=False))

In [ ]:
###############################
    #VIF-анализ - панельные данные спецификация в первых разностях 2 спецификация
###############################

vif_variables = [
    'd_Int_Rate_ConsCred_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_ConsCred_adj',
   # 'd_ln_New_Loans_Progr',
    'Mon_Shock',
    'd_ln_New_Loans_ConsCred_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_Exc_rate_adj',
    'Inflation_Expectations_adj'                   
]

df_vif = df_exog[vif_variables].copy()
df_vif = df_vif.dropna()

X_vif = sm.add_constant(df_vif[vif_variables])

# Расчет VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = vif_variables
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i+1) for i in range(len(vif_variables))]
vif_data = vif_data.sort_values('VIF', ascending=False)

print("\n" + "="*60)
print("РЕЗУЛЬТАТЫ VIF АНАЛИЗА - ПАНЕЛЬНЫЕ ДАННЫЕ В ПЕРВЫХ РАЗНОСТЯХ СПЕЦИФИКАЦИЯ 2")
print("="*60)
print(vif_data.to_string(index=False))

In [ ]:
###############################
    #VIF-анализ - панельные данные спецификация в первых разностях 3 спецификация
###############################

vif_variables = [
    'd_Int_Rate_Mort_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_Mort_adj',
    #'d_ln_New_Loans_Progr',
    'd_ln_New_Loans_Mort_adj',
    'Mon_Shock',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_Exc_rate_adj',
    'Inflation_Expectations_adj'                     
]

df_vif = df_exog[vif_variables].copy()
df_vif = df_vif.dropna()

X_vif = sm.add_constant(df_vif[vif_variables])

# Расчет VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = vif_variables
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i+1) for i in range(len(vif_variables))]
vif_data = vif_data.sort_values('VIF', ascending=False)

print("\n" + "="*60)
print("РЕЗУЛЬТАТЫ VIF АНАЛИЗА - ПАНЕЛЬНЫЕ ДАННЫЕ В ПЕРВЫХ РАЗНОСТЯХ СПЕЦИФИКАЦИЯ 3")
print("="*60)
print(vif_data.to_string(index=False))

In [ ]:
###############################
    #Тест на стационарность I(1) рядов
###############################

custom_vars = [
    'd_Int_Rate_FL_adj',
    'd_Int_Rate_FL_lag1_adj',
    'd_Int_Rate_Mort_adj',
    'Credit_impulse_adj',
    'd_Int_Rate_Mort_lag1_adj',
    'd_Int_Rate_ConsCred_adj',
    'd_Int_Rate_ConsCred_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_Fl_adj',
    'd_Def_Zadolg_Mort_adj',
    'd_Def_Zadolg_ConsCred_adj',
    'd_ln_New_Loans_Fl_adj',         
    'd_ln_New_Loans_Mort_adj',       
    'd_ln_New_Loans_ConsCred_adj',   
    'Inflation_Expectations_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'Mon_Shock',
    'd_Exc_rate_adj'
]

# Запуск функции с пользовательским списком переменных
results_seasonal_adjusted = run_panel_unit_root_tests_python(df_exog, panel_vars=custom_vars)
print("\n" + "="*70)
print("ЗАПУСК ТЕСТОВ НА ЕДИНИЧНЫЕ КОРНИ ДЛЯ I(1) РЯДОВ")
print("="*70)

In [ ]:
df_exog.info()

In [ ]:
###############################
# Построение линейных моделей на панельных данных
# (d_Int_Rate_FL зависимая переменная)
###############################

print("="*70)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE")
print("Зависимая переменная: d_Int_Rate_FL")
print("="*70)

exog_vars_initial = [
    'd_Int_Rate_FL_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_Fl_adj',
   # 'd_ln_New_Loans_Progr',
    'Mon_Shock',
    'd_ln_New_Loans_Fl_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_exc_rate_adj',
    'Inflation_Expectations_adj'                          
]

dependent_var = 'd_Int_Rate_FL_adj'

# Создаем копию df_exog для работы
df_clean = df_exog.copy()

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
print("\n" + "="*70)
print("МОДЕЛЬ 1: POOLED OLS")
print("="*70)

try:
    pooled_mod = PooledOLS(y, X)
    pooled_res = pooled_mod.fit(cov_type='clustered')
    print(pooled_res.summary)
    pooled_success = True
except Exception as e:
    print(f"ERROR: {e}")
    pooled_success = False

# ===== FIXED EFFECTS =====
print("\n" + "="*70)
print("МОДЕЛЬ 2: FIXED EFFECTS (WITHIN)")
print("="*70)

try:
    fe_mod = PanelOLS(y, X, entity_effects=True)
    fe_res = fe_mod.fit(cov_type='clustered', cluster_entity=True)
    print(fe_res.summary)
    fe_success = True
except Exception as e:
    print(f"ERROR: {e}")
    fe_success = False

# ===== RANDOM EFFECTS =====
print("\n" + "="*70)
print("МОДЕЛЬ 3: RANDOM EFFECTS")
print("="*70)

try:
    re_mod = RandomEffects(y, X)
    re_res = re_mod.fit(cov_type='clustered')
    print(re_res.summary)
    re_success = True
except Exception as e:
    print(f"ERROR: {e}")
    re_success = False

In [ ]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)

# ТЕСТ ХАУСМАНА
print("\n ТЕСТ ХАУСМАНА (FE vs RE)")
print("-"*70)

if fe_success and re_success:
    try:
        coef_diff = (fe_res.params - re_res.params).dropna()
        
        var_fe = fe_res.cov.loc[coef_diff.index, coef_diff.index]
        var_re = re_res.cov.loc[coef_diff.index, coef_diff.index]
        var_diff = var_fe - var_re
        
        try:
            inv_var_diff = np.linalg.inv(var_diff.values)
        except np.linalg.LinAlgError:
            print("Матрица сингулярна, используется pseudo-inverse")
            inv_var_diff = np.linalg.pinv(var_diff.values)
        
        H = coef_diff.values @ inv_var_diff @ coef_diff.values
        p_val = 1 - chi2.cdf(H, df=len(coef_diff))
        
        print(f"H-статистика: {H:.4f}")
        print(f"p-значение: {p_val:.6f}")
        print(f"df: {len(coef_diff)}")
        
        if p_val < 0.05:
            print("\nВывод: p < 0.05 - Используйте FIXED EFFECTS")
        else:
            print("\nВывод: p >= 0.05 - Используйте RANDOM EFFECTS")
    except Exception as e:
        print(f"ERROR: {e}")
else:
    print("Невозможно провести (требуются обе модели)")

# ТЕСТ БРЕУША-ПАГАНА
print("\n ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)")
print("-"*70)

if re_success and pooled_success:
    try:
        u_pooled = (y.values - X.values @ pooled_res.params.values.reshape(-1, 1)).flatten()
        sigma_u_sq = (u_pooled**2).sum() / len(u_pooled)
        
        regions = df_clean.index.get_level_values('Region').unique()
        N = len(regions)
        T = len(df_clean) // N
        
        sum_mean_u_sq = 0
        for region in regions:
            u_region = u_pooled[df_clean.index.get_level_values('Region') == region]
            mean_u = u_region.mean()
            sum_mean_u_sq += mean_u**2
        
        LM = (N * T**2) / (2 * (T - 1)) * (sum_mean_u_sq / (sigma_u_sq * N) - 1)**2
        p_val_bp = 1 - chi2.cdf(LM, df=1)
        
        print(f"LM статистика: {LM:.4f}")
        print(f"p-значение: {p_val_bp:.6f}")
        print(f"N (регионов): {N}, T (периодов): {T}")
        
        if p_val_bp < 0.05:
            print("\nВывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)")
        else:
            print("\nВывод: p >= 0.05 - Нет эффектов (адекватна Pooled)")
    except Exception as e:
        print(f"ERROR: {e}")
else:
    print("Невозможно провести")

# F-ТЕСТ
print("\n F-ТЕСТ (FE vs Pooled)")
print("-"*70)

if fe_success and pooled_success:
    try:
        N = df_clean.index.get_level_values('Region').nunique()
        T = df_clean.index.get_level_values('Date').nunique()
        k = len(exog_vars_for_regression)
        
        u_pooled = (y.values - X.values @ pooled_res.params.values.reshape(-1, 1)).flatten()
        u_fe = (y.values - X.values @ fe_res.params.values.reshape(-1, 1)).flatten()
        
        SSR_pooled = (u_pooled**2).sum()
        SSR_fe = (u_fe**2).sum()
        
        F_stat = ((SSR_pooled - SSR_fe) / (N - 1)) / (SSR_fe / (N*T - N - k))
        p_val_f = 1 - f.cdf(F_stat, N-1, N*T - N - k)
        
        print(f"F-статистика: {F_stat:.4f}")
        print(f"p-значение: {p_val_f:.6f}")
        print(f"df: ({N-1}, {N*T - N - k})")
        
        if p_val_f < 0.05:
            print("\nВывод: p < 0.05 - FE значимо лучше чем Pooled")
        else:
            print("\nВывод: p >= 0.05 - Pooled адекватна")
    except Exception as e:
        print(f"ERROR: {e}")
else:
    print("Невозможно провести")

In [ ]:
###############################
# Построение линейных моделей на панельных данных
# (d_Int_Rate_ConsCred зависимая переменная)
###############################

print("="*60)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ POOL + КЛАСТЕРИЗОВАННЫЕ SE; ЗАВИСИМАЯ ПЕРЕМЕННАЯ d_Int_Rate_ConsCred")
print("="*60)

exog_vars_initial = [
    'd_Int_Rate_ConsCred_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_ConsCred_adj',
    #'d_ln_New_Loans_Progr',
    'Mon_Shock',
    'd_ln_New_Loans_ConsCred_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_Exc_rate_adj',
    'Inflation_Expectations_adj'                          
]

dependent_var = 'd_Int_Rate_ConsCred_adj'

# Создаем копию df_exog для работы
df_clean = df_exog.copy()

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
print("\n" + "="*70)
print("МОДЕЛЬ 1: POOLED OLS")
print("="*70)

try:
    pooled_mod = PooledOLS(y, X)
    pooled_res = pooled_mod.fit(cov_type='clustered')
    print(pooled_res.summary)
    pooled_success = True
except Exception as e:
    print(f"ERROR: {e}")
    pooled_success = False

# ===== FIXED EFFECTS =====
print("\n" + "="*70)
print("МОДЕЛЬ 2: FIXED EFFECTS (WITHIN)")
print("="*70)

try:
    fe_mod = PanelOLS(y, X, entity_effects=True)
    fe_res = fe_mod.fit(cov_type='clustered', cluster_entity=True)
    print(fe_res.summary)
    fe_success = True
except Exception as e:
    print(f"ERROR: {e}")
    fe_success = False

# ===== RANDOM EFFECTS =====
print("\n" + "="*70)
print("МОДЕЛЬ 3: RANDOM EFFECTS")
print("="*70)

try:
    re_mod = RandomEffects(y, X)
    re_res = re_mod.fit(cov_type='clustered')
    print(re_res.summary)
    re_success = True
except Exception as e:
    print(f"ERROR: {e}")
    re_success = False

In [ ]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)

# ТЕСТ ХАУСМАНА
print("\n ТЕСТ ХАУСМАНА (FE vs RE)")
print("-"*70)

if fe_success and re_success:
    try:
        coef_diff = (fe_res.params - re_res.params).dropna()
        
        var_fe = fe_res.cov.loc[coef_diff.index, coef_diff.index]
        var_re = re_res.cov.loc[coef_diff.index, coef_diff.index]
        var_diff = var_fe - var_re
        
        try:
            inv_var_diff = np.linalg.inv(var_diff.values)
        except np.linalg.LinAlgError:
            print("Матрица сингулярна, используется pseudo-inverse")
            inv_var_diff = np.linalg.pinv(var_diff.values)
        
        H = coef_diff.values @ inv_var_diff @ coef_diff.values
        p_val = 1 - chi2.cdf(H, df=len(coef_diff))
        
        print(f"H-статистика: {H:.4f}")
        print(f"p-значение: {p_val:.6f}")
        print(f"df: {len(coef_diff)}")
        
        if p_val < 0.05:
            print("\nВывод: p < 0.05 - Используйте FIXED EFFECTS")
        else:
            print("\nВывод: p >= 0.05 - Используйте RANDOM EFFECTS")
    except Exception as e:
        print(f"ERROR: {e}")
else:
    print("Невозможно провести (требуются обе модели)")

# ТЕСТ БРЕУША-ПАГАНА
print("\n ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)")
print("-"*70)

if re_success and pooled_success:
    try:
        u_pooled = (y.values - X.values @ pooled_res.params.values.reshape(-1, 1)).flatten()
        sigma_u_sq = (u_pooled**2).sum() / len(u_pooled)
        
        regions = df_clean.index.get_level_values('Region').unique()
        N = len(regions)
        T = len(df_clean) // N
        
        sum_mean_u_sq = 0
        for region in regions:
            u_region = u_pooled[df_clean.index.get_level_values('Region') == region]
            mean_u = u_region.mean()
            sum_mean_u_sq += mean_u**2
        
        LM = (N * T**2) / (2 * (T - 1)) * (sum_mean_u_sq / (sigma_u_sq * N) - 1)**2
        p_val_bp = 1 - chi2.cdf(LM, df=1)
        
        print(f"LM статистика: {LM:.4f}")
        print(f"p-значение: {p_val_bp:.6f}")
        print(f"N (регионов): {N}, T (периодов): {T}")
        
        if p_val_bp < 0.05:
            print("\nВывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)")
        else:
            print("\nВывод: p >= 0.05 - Нет эффектов (адекватна Pooled)")
    except Exception as e:
        print(f"ERROR: {e}")
else:
    print("Невозможно провести")

# F-ТЕСТ
print("\n F-ТЕСТ (FE vs Pooled)")
print("-"*70)

if fe_success and pooled_success:
    try:
        N = df_clean.index.get_level_values('Region').nunique()
        T = df_clean.index.get_level_values('Date').nunique()
        k = len(exog_vars_for_regression)
        
        u_pooled = (y.values - X.values @ pooled_res.params.values.reshape(-1, 1)).flatten()
        u_fe = (y.values - X.values @ fe_res.params.values.reshape(-1, 1)).flatten()
        
        SSR_pooled = (u_pooled**2).sum()
        SSR_fe = (u_fe**2).sum()
        
        F_stat = ((SSR_pooled - SSR_fe) / (N - 1)) / (SSR_fe / (N*T - N - k))
        p_val_f = 1 - f.cdf(F_stat, N-1, N*T - N - k)
        
        print(f"F-статистика: {F_stat:.4f}")
        print(f"p-значение: {p_val_f:.6f}")
        print(f"df: ({N-1}, {N*T - N - k})")
        
        if p_val_f < 0.05:
            print("\nВывод: p < 0.05 - FE значимо лучше чем Pooled")
        else:
            print("\nВывод: p >= 0.05 - Pooled адекватна")
    except Exception as e:
        print(f"ERROR: {e}")
else:
    print("Невозможно провести")

In [ ]:
###############################
# Построение линейных моделей на панельных данных
# (d_Int_Rate_Mort зависимая переменная)
###############################

print("="*60)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ POOL + КЛАСТЕРИЗОВАННЫЕ SE; ЗАВИСИМАЯ ПЕРЕМЕННАЯ d_Int_Rate_Mort")
print("="*60)

exog_vars_initial = [
    'd_Int_Rate_Mort_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_Mort_adj',
   # 'd_ln_New_Loans_Progr',
    'd_ln_New_Loans_Mort_adj',
    'Mon_Shock',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_Exc_rate_adj',
    'Inflation_Expectations_adj'                        
]

dependent_var = 'd_Int_Rate_Mort_adj'


# Создаем копию df_exog для работы
df_clean = df_exog.copy()

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
print("\n" + "="*70)
print("МОДЕЛЬ 1: POOLED OLS")
print("="*70)

try:
    pooled_mod = PooledOLS(y, X)
    pooled_res = pooled_mod.fit(cov_type='clustered')
    print(pooled_res.summary)
    pooled_success = True
except Exception as e:
    print(f"ERROR: {e}")
    pooled_success = False

# ===== FIXED EFFECTS =====
print("\n" + "="*70)
print("МОДЕЛЬ 2: FIXED EFFECTS (WITHIN)")
print("="*70)

try:
    fe_mod = PanelOLS(y, X, entity_effects=True)
    fe_res = fe_mod.fit(cov_type='clustered', cluster_entity=True)
    print(fe_res.summary)
    fe_success = True
except Exception as e:
    print(f"ERROR: {e}")
    fe_success = False

# ===== RANDOM EFFECTS =====
print("\n" + "="*70)
print("МОДЕЛЬ 3: RANDOM EFFECTS")
print("="*70)

try:
    re_mod = RandomEffects(y, X)
    re_res = re_mod.fit(cov_type='clustered')
    print(re_res.summary)
    re_success = True
except Exception as e:
    print(f"ERROR: {e}")
    re_success = False

In [ ]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)

# ТЕСТ ХАУСМАНА
print("\n ТЕСТ ХАУСМАНА (FE vs RE)")
print("-"*70)

if fe_success and re_success:
    try:
        coef_diff = (fe_res.params - re_res.params).dropna()
        
        var_fe = fe_res.cov.loc[coef_diff.index, coef_diff.index]
        var_re = re_res.cov.loc[coef_diff.index, coef_diff.index]
        var_diff = var_fe - var_re
        
        try:
            inv_var_diff = np.linalg.inv(var_diff.values)
        except np.linalg.LinAlgError:
            print("Матрица сингулярна, используется pseudo-inverse")
            inv_var_diff = np.linalg.pinv(var_diff.values)
        
        H = coef_diff.values @ inv_var_diff @ coef_diff.values
        p_val = 1 - chi2.cdf(H, df=len(coef_diff))
        
        print(f"H-статистика: {H:.4f}")
        print(f"p-значение: {p_val:.6f}")
        print(f"df: {len(coef_diff)}")
        
        if p_val < 0.05:
            print("\nВывод: p < 0.05 - Используйте FIXED EFFECTS")
        else:
            print("\nВывод: p >= 0.05 - Используйте RANDOM EFFECTS")
    except Exception as e:
        print(f"ERROR: {e}")
else:
    print("Невозможно провести (требуются обе модели)")

# ТЕСТ БРЕУША-ПАГАНА
print("\n ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)")
print("-"*70)

if re_success and pooled_success:
    try:
        u_pooled = (y.values - X.values @ pooled_res.params.values.reshape(-1, 1)).flatten()
        sigma_u_sq = (u_pooled**2).sum() / len(u_pooled)
        
        regions = df_clean.index.get_level_values('Region').unique()
        N = len(regions)
        T = len(df_clean) // N
        
        sum_mean_u_sq = 0
        for region in regions:
            u_region = u_pooled[df_clean.index.get_level_values('Region') == region]
            mean_u = u_region.mean()
            sum_mean_u_sq += mean_u**2
        
        LM = (N * T**2) / (2 * (T - 1)) * (sum_mean_u_sq / (sigma_u_sq * N) - 1)**2
        p_val_bp = 1 - chi2.cdf(LM, df=1)
        
        print(f"LM статистика: {LM:.4f}")
        print(f"p-значение: {p_val_bp:.6f}")
        print(f"N (регионов): {N}, T (периодов): {T}")
        
        if p_val_bp < 0.05:
            print("\nВывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)")
        else:
            print("\nВывод: p >= 0.05 - Нет эффектов (адекватна Pooled)")
    except Exception as e:
        print(f"ERROR: {e}")
else:
    print("Невозможно провести")

# F-ТЕСТ
print("\n F-ТЕСТ (FE vs Pooled)")
print("-"*70)

if fe_success and pooled_success:
    try:
        N = df_clean.index.get_level_values('Region').nunique()
        T = df_clean.index.get_level_values('Date').nunique()
        k = len(exog_vars_for_regression)
        
        u_pooled = (y.values - X.values @ pooled_res.params.values.reshape(-1, 1)).flatten()
        u_fe = (y.values - X.values @ fe_res.params.values.reshape(-1, 1)).flatten()
        
        SSR_pooled = (u_pooled**2).sum()
        SSR_fe = (u_fe**2).sum()
        
        F_stat = ((SSR_pooled - SSR_fe) / (N - 1)) / (SSR_fe / (N*T - N - k))
        p_val_f = 1 - f.cdf(F_stat, N-1, N*T - N - k)
        
        print(f"F-статистика: {F_stat:.4f}")
        print(f"p-значение: {p_val_f:.6f}")
        print(f"df: ({N-1}, {N*T - N - k})")
        
        if p_val_f < 0.05:
            print("\nВывод: p < 0.05 - FE значимо лучше чем Pooled")
        else:
            print("\nВывод: p >= 0.05 - Pooled адекватна")
    except Exception as e:
        print(f"ERROR: {e}")
else:
    print("Невозможно провести")

In [ ]:
df_clean.head()

###############################
# Построение PVAR S-GMM моделей
###############################
from pydynpd import regression

# Подготовка данных
df_for_gmm = df_clean.reset_index()
df_for_gmm = df_for_gmm.sort_values(['Region', 'Date'])

# System GMM 

# Используем спецификацию, которая работала ранее
command_str_system = (
    'd_Int_Rate_FL_adj L1.d_Int_Rate_FL_adj '
    'd_Cred_nagr_adj d_D_top5_rozn_adj '
    'Mon_Shock d_Bonds_Rate_Correct_5Y_adj d_Exc_rate_adj | '
    'gmm(d_Int_Rate_FL_adj, 2:4) '
    'iv(d_Cred_nagr_adj d_D_top5_rozn_adj Mon_Shock d_Bonds_Rate_Correct_5Y_adj d_Exc_rate_adj) | '
    'collapse'
)

mydpd_system = regression.abond(command_str_system, df_for_gmm, ['Region', 'Date'])
